In [1]:
import math
import pandas as pd
from collections import Counter

# ----------------------------
# توابع پایه (طبق جزوه)
# ----------------------------

def tf_log(tf):
    return 1 + math.log10(tf) if tf > 0 else 0

def idf(df, N):
    return math.log10(N / df) if df > 0 else 0

def l2_norm(vec):
    return math.sqrt(sum(v ** 2 for v in vec))

# ----------------------------
# تابع اصلی (داینامیک)
# ----------------------------

def tf_idf_table(query_terms, doc_terms, df_dict, N):
    """
    query_terms : list[str]
    doc_terms   : list[str]
    df_dict     : dict(term -> df)
    N           : int (تعداد کل اسناد)
    """

    q_tf = Counter(query_terms)
    d_tf = Counter(doc_terms)

    vocabulary = sorted(set(query_terms) | set(doc_terms))

    rows = []

    for t in vocabulary:
        q_tf_raw = q_tf.get(t, 0)
        d_tf_raw = d_tf.get(t, 0)

        q_tf_wt = tf_log(q_tf_raw)
        d_tf_wt = tf_log(d_tf_raw)

        df_t = df_dict.get(t, 1)  # اگر نبود، حداقل 1
        idf_t = idf(df_t, N)

        q_wt = q_tf_wt * idf_t
        d_wt = d_tf_wt * idf_t

        rows.append([
            t,
            q_tf_raw,
            round(q_tf_wt, 3),
            df_t,
            round(idf_t, 3),
            round(q_wt, 3),
            0,  # Query normalize
            d_tf_raw,
            round(d_tf_wt, 3),
            round(d_wt, 3),
            0,  # Doc normalize
            0   # Product
        ])

    # ----------------------------
    # نرمال‌سازی
    # ----------------------------

    q_norm = l2_norm([r[5] for r in rows])
    d_norm = l2_norm([r[9] for r in rows])

    for r in rows:
        r[6] = round(r[5] / q_norm, 3) if q_norm != 0 else 0
        r[10] = round(r[9] / d_norm, 3) if d_norm != 0 else 0
        r[11] = round(r[6] * r[10], 3)

    columns = [
        "Term",
        "Query tf-raw",
        "Query tf-wt",
        "Query df",
        "Query idf",
        "Query wt",
        "Query n'lize",
        "Doc tf-raw",
        "Doc tf-wt",
        "Doc wt",
        "Doc n'lize",
        "Prod (Query × Doc)"
    ]

    return pd.DataFrame(rows, columns=columns)


In [2]:
query = ["best", "car", "insurance"]
document = ["auto", "car", "insurance", "insurance"]

df_values = {
    "auto": 5000,
    "best": 50000,
    "car": 10000,
    "insurance": 1000
}

N = 1_000_000

table = tf_idf_table(query, document, df_values, N)
print(table)
print("\nCosine Similarity =", table["Prod (Query × Doc)"].sum())


        Term  Query tf-raw  Query tf-wt  Query df  Query idf  Query wt  \
0       auto             0          0.0      5000      2.301     0.000   
1       best             1          1.0     50000      1.301     1.301   
2        car             1          1.0     10000      2.000     2.000   
3  insurance             1          1.0      1000      3.000     3.000   

   Query n'lize  Doc tf-raw  Doc tf-wt  Doc wt  Doc n'lize  Prod (Query × Doc)  
0         0.000           1      1.000   2.301       0.465               0.000  
1         0.339           0      0.000   0.000       0.000               0.000  
2         0.522           1      1.000   2.000       0.404               0.211  
3         0.783           2      1.301   3.903       0.788               0.617  

Cosine Similarity = 0.828
